In [53]:
import pandas as pd
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
import seaborn as sns
from icu_sleep_breathing_2020_help_functions import *

In [54]:
[summary_subjects_icu, summary_subjects_sleeplab, 
 summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria()
study_ids = summary_days_icu.study_id.unique()
print(study_ids.shape)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3254: DtypeWarning: Columns (14,76,77,78,5683,5684,5685,11290,11291,11292,16899,16900,16901,18798,18799,18800,18826,18827,18828,18833,18834,18835,22506,22507,22508,28113,28114,28115,33722,33723,33724,39329,39330,39331,41130,41131,41132,41144,41145,41146,41158,41159,41160,41228,41229,41230,41235,41236,41237,41242,41243,41244,41249,41250,41251,41256,41257,41258,41263,41264,41265,44936,44937,44938,46858,46859,46860,50491,50545,50546,50547,52353,52354,52355,52367,52368,52369,52451,52452,52453,56098,56152,56153,56154,57960,57961,57962,57967,57968,57969,57988,57989,57990,58044,58045,58046,58051,58052,58053,58058,58059,58060,58065,58066,58067,58079,58080,58081,58086,58087,58088,61705,61759,61760,61761,63569,63570,63571,63583,63584,63585,63653,63654,63655,63667,63668,63669,67314,67368,67369,67370,69176,69177,69178,69190,69191,69192,69204,69205,69206,69260,69261,69262,69281,69282,69283,69295,69

# of subjects ICU before inclusion criteria: 108
# of 12-hour segments ICU before inclusion criteria: 1230
# of subjects ICU after inclusion criteria: 103
# of 12-hour segments ICU after inclusion criteria: 621
24-hour segments: 256
12-hour segments: 365

# of subjects sleeplab before inclusion criteria: 412
# of 12-hour segments sleeplab before inclusion criteria: 412
# of subjects sleeplab after inclusion criteria: 307
# of 12-hour segments sleeplab after before inclusion criteria: 307
(103,)


In [3]:
assert summary_subjects_icu.study_id.unique().shape[0] == 103, "Currently, we have 103 subjects included!"

In [4]:
if 0:
    dir_summaries = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing'
    summary_subjects_icu_full = pd.read_csv(os.path.join(dir_summaries, 'summary_subjects_icu.csv'))
    summary_subjects_icu_full['inclusion_sleepbreathing'] = 0
    summary_subjects_icu_full.loc[np.isin(summary_subjects_icu_full.study_id.values, summary_subjects_icu.study_id.values), 'inclusion_sleepbreathing'] = 1
    print(summary_subjects_icu_full['inclusion_sleepbreathing'].sum())
    print(summary_subjects_icu_full['inclusion_apnea'].sum())

In [55]:
summary_subjects_sleeplab.matched_ahi_0_5.sum()

77.0

### from here, the matching pipeline begins. first, select the sleeplab cohort you want to use for matching (all, or stratified by ahi.)


In [5]:
def select_ahi_cohort(summary_days_sleeplab_all, sleeplab_cohort):
    
    if sleeplab_cohort == 'all':
        summary_days_sleeplab = summary_days_sleeplab_all.copy()
    elif sleeplab_cohort == 'ahi_0_5':
        summary_days_sleeplab = summary_days_sleeplab_all.loc[summary_days_sleeplab_all.ahi_expert.apply(lambda x: x < 5)].copy()
    elif sleeplab_cohort == 'ahi_5_15':
        summary_days_sleeplab = summary_days_sleeplab_all.loc[summary_days_sleeplab_all.ahi_expert.apply(lambda x: 5 <= x < 15)].copy()
    elif sleeplab_cohort == 'ahi_15_30':
        summary_days_sleeplab = summary_days_sleeplab_all.loc[summary_days_sleeplab_all.ahi_expert.apply(lambda x: 15 <= x < 30)].copy()
    elif sleeplab_cohort == 'ahi_30_100':
        summary_days_sleeplab = summary_days_sleeplab_all.loc[summary_days_sleeplab_all.ahi_expert.apply(lambda x: 30 <= x)].copy()
    elif sleeplab_cohort == 'ahi_15_100':
        summary_days_sleeplab = summary_days_sleeplab_all.loc[summary_days_sleeplab_all.ahi_expert.apply(lambda x: 15 <= x)].copy()

    
    return summary_days_sleeplab

In [6]:
summary_subjects_sleeplab_all = summary_subjects_sleeplab.copy()
summary_days_sleeplab_all = summary_days_sleeplab.copy()

In [7]:
ahi_categories = ['all', 'ahi_0_5', 'ahi_5_15', 'ahi_15_30', 'ahi_30_100', 'ahi_15_100']

for sleeplab_cohort in ahi_categories:
    summary_days_sleeplab_all['matched_' + sleeplab_cohort] = 0

if 1:
    for sleeplab_cohort in ahi_categories:
        summary_days_sleeplab = select_ahi_cohort(summary_days_sleeplab_all, sleeplab_cohort)
        print(f'AHI selection: {sleeplab_cohort}. # subjects: {summary_days_sleeplab.shape[0]}')

AHI selection: all. # subjects: 307
AHI selection: ahi_0_5. # subjects: 136
AHI selection: ahi_5_15. # subjects: 104
AHI selection: ahi_15_30. # subjects: 49
AHI selection: ahi_30_100. # subjects: 18
AHI selection: ahi_15_100. # subjects: 67


In [92]:
# from here on, i manually loop through the ahi categories.

sleeplab_cohort = 'ahi_15_100'
assert sleeplab_cohort in ['all', 'ahi_0_5', 'ahi_5_15', 'ahi_15_30', 'ahi_30_100', 'ahi_15_100']

summary_days_sleeplab = select_ahi_cohort(summary_days_sleeplab_all, sleeplab_cohort)

In [93]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

def get_matching_pairs(treated_df, non_treated_df, scaler=True, n_neighbors=1, scale_var1=1):

    treated_x = treated_df.values
    non_treated_x = non_treated_df.values
    
# normality-approach:
#     treated_zscores = StandardScaler().fit(treated_x).transform(treated_x)
#     treated_x = treated_x[np.all(abs(treated_zscores) < 2.5, axis=1)]
# quantile approach:
    if 0:
        q0 = np.quantile(treated_x,0.0225, axis=0)
        q1 = np.quantile(treated_x,0.975, axis=0)
        treated_x = treated_x[np.all((treated_x >= q0) & (treated_x <= q1), axis=1)]

    if scaler == True:
        print('scaler on.')
        scaler = StandardScaler()
        scaler.fit(treated_x)
        treated_x = scaler.transform(treated_x)
        treated_x[:,0] = treated_x[:,0]/scale_var1
        non_treated_x = scaler.transform(non_treated_x)
            
    nbrs = NearestNeighbors(n_neighbors=n_neighbors,radius=1000000, algorithm='ball_tree', p=1).fit(non_treated_x)
    distances, indices = nbrs.kneighbors(treated_x)
    indices = indices.flatten().reshape(indices.shape[0]*n_neighbors)
    indices = np.unique(indices)
    matched = non_treated_df.iloc[indices]
    return matched

### check distributions

In [94]:
icu_male = summary_subjects_icu[summary_subjects_icu.sex=='Male']
icu_female = summary_subjects_icu[summary_subjects_icu.sex=='Female']

lab_male = summary_days_sleeplab[summary_days_sleeplab.sex=='Male']
lab_female = summary_days_sleeplab[summary_days_sleeplab.sex=='Female']


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/pandas/core/ops/__init__.py:1115: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = method(y)


In [95]:
demographics_icu = summary_subjects_icu[['study_id', 'mrn', 'age', 'sex']]
demographics_sleeplab = summary_days_sleeplab[['study_id', 'mrn', 'age', 'sex']]
demographics_sleeplab['sex'] = (demographics_sleeplab['sex']=='Male').values
demographics_icu['population'] = 'ICU'
demographics_sleeplab['population'] = 'Sleeplab'

demographics = pd.concat([demographics_sleeplab, demographics_icu], axis=0)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning

In [96]:
num_sleeplab_female = demographics[(demographics.population=='Sleeplab') & (demographics.sex==0)].shape[0]
num_sleeplab_male = demographics[(demographics.population=='Sleeplab') & (demographics.sex==1)].shape[0]
num_icu_female = demographics[(demographics.population=='ICU') & (demographics.sex==0)].shape[0]
num_icu_male = demographics[(demographics.population=='ICU') & (demographics.sex==1)].shape[0]

print(f'Sleeplab Female: {num_sleeplab_female}')
print(f'Sleeplab Male: {num_sleeplab_male}')
print(f'ICU Female: {num_icu_female}')
print(f'ICU Male: {num_icu_male}')

print(f'ICU Male/Female Ratio: {num_icu_male/num_icu_female:.1f}')
print(f'Sleeplab Male/Female Ratio: {num_sleeplab_male/num_sleeplab_female:.1f}')



Sleeplab Female: 18
Sleeplab Male: 49
ICU Female: 41
ICU Male: 62
ICU Male/Female Ratio: 1.5
Sleeplab Male/Female Ratio: 2.7


In [97]:
plt.figure(figsize=(10,3))
plt.subplot(131)
plt.title('All Subjects')
ax = sns.boxplot(x='population', y='age', data=demographics, width=0.9, fliersize=0)
ax = sns.swarmplot(x='population', y='age', data=demographics, color='red', edgecolor='black', linewidth=1, size=2)
ax.set_ylim([20,105])
plt.subplot(132)
plt.title('Female Subjects')
ax = sns.boxplot(x='population', y='age', data=demographics[demographics.sex==0], width=0.9, fliersize=0)
ax = sns.swarmplot(x='population', y='age', data=demographics[demographics.sex==0], color='red', edgecolor='black', linewidth=1, size=2)
ax.set_ylim([20,105])

plt.subplot(133)
plt.title('Male Subjects')
ax = sns.boxplot(x='population', y='age', data=demographics[demographics.sex==1], width=0.9, fliersize=0)
ax = sns.swarmplot(x='population', y='age', data=demographics[demographics.sex==1], color='red', edgecolor='black', linewidth=1, size=2)
ax.set_ylim([20,105])
plt.tight_layout()

print(f"Median Age for ICU female: {demographics.loc[(demographics.sex==0) & (demographics.population=='ICU'), 'age'].median():.1f}")
print(f"Median Age for sleeplab female: {demographics.loc[(demographics.sex==0) & (demographics.population=='Sleeplab'), 'age'].median():.1f}")
print('')
print(f"Median Age for ICU male: {demographics.loc[(demographics.sex==1) & (demographics.population=='ICU'), 'age'].median():.1f}")
print(f"Median Age for sleeplab male: {demographics.loc[(demographics.sex==1) & (demographics.population=='Sleeplab'), 'age'].median():.1f}")
print('')


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Median Age for ICU female: 67.0
Median Age for sleeplab female: 71.8

Median Age for ICU male: 68.0
Median Age for sleeplab male: 60.6



In [98]:
demographics_original = demographics.copy()

In [99]:
demographics = demographics_original.copy()

In [100]:
plt.figure(figsize=(10,3))
plt.subplot(131)
plt.title('All Subjects')
ax = sns.boxplot(x='population', y='age', data=demographics, width=0.9, fliersize=0)
ax = sns.swarmplot(x='population', y='age', data=demographics, color='red', edgecolor='black', linewidth=1, size=2)
ax.set_ylim([20,105])
plt.subplot(132)
plt.title('Female Subjects')
ax = sns.boxplot(x='population', y='age', data=demographics[demographics.sex==0], width=0.9, fliersize=0)
ax = sns.swarmplot(x='population', y='age', data=demographics[demographics.sex==0], color='red', edgecolor='black', linewidth=1, size=2)
ax.set_ylim([20,105])

plt.subplot(133)
plt.title('Male Subjects')
ax = sns.boxplot(x='population', y='age', data=demographics[demographics.sex==1], width=0.9, fliersize=0)
ax = sns.swarmplot(x='population', y='age', data=demographics[demographics.sex==1], color='red', edgecolor='black', linewidth=1, size=2)
ax.set_ylim([20,105])
plt.tight_layout()

print(f"Median Age for ICU female: {demographics.loc[(demographics.sex==0) & (demographics.population=='ICU'), 'age'].median():.1f}")
print(f"Median Age for sleeplab female: {demographics.loc[(demographics.sex==0) & (demographics.population=='Sleeplab'), 'age'].median():.1f}")
print('')
print(f"Median Age for ICU male: {demographics.loc[(demographics.sex==1) & (demographics.population=='ICU'), 'age'].median():.1f}")
print(f"Median Age for sleeplab male: {demographics.loc[(demographics.sex==1) & (demographics.population=='Sleeplab'), 'age'].median():.1f}")
print('')


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Median Age for ICU female: 67.0
Median Age for sleeplab female: 71.8

Median Age for ICU male: 68.0
Median Age for sleeplab male: 60.6



In [101]:
from scipy.stats import ttest_ind, mannwhitneyu

In [102]:
variables = ['age','sex']

scale_var1 = 1
if sleeplab_cohort == 'all':
    n_neighbors = 9
elif sleeplab_cohort == 'ahi_0_5':
    n_neighbors = 3
elif sleeplab_cohort == 'ahi_5_15':
    n_neighbors = 3
elif sleeplab_cohort == 'ahi_15_30':
    n_neighbors = 3
elif sleeplab_cohort == 'ahi_30_100':
    n_neighbors = 10
elif sleeplab_cohort == 'ahi_15_100':
    n_neighbors = 5
    
print(f'n_neighbor: {n_neighbors}')
demographics.age = np.round(demographics.age,1)

# demographics['age'] = demographics['age'].astype(float).values/scale_age
demographics.reset_index(drop=True, inplace=True)

icu_sleep_airgo_demographics = demographics[demographics.population=='ICU']
sleeplab_demographics = demographics[demographics.population=='Sleeplab']

to_match_df = icu_sleep_airgo_demographics[variables].copy()
to_match_df.age = to_match_df.age + 3
matched_df = get_matching_pairs(to_match_df, sleeplab_demographics[variables], n_neighbors=n_neighbors, 
                                scaler=True, scale_var1=scale_var1)

print(f'No. of matches: {matched_df.shape[0]}')

idx_keep = np.concatenate([np.array(matched_df.index), np.array(icu_sleep_airgo_demographics.index)])
no_matched = len(matched_df.index)

if sleeplab_cohort == 'ahi_30_100':
    print('No Matching for AHI>30')
else:
    demographics = demographics.loc[idx_keep]
demographics.shape
# demographics['age'] = demographics['age']*scale_age


n_neighbor: 5
scaler on.
No. of matches: 52


(155, 5)

In [103]:
plt.figure(figsize=(10,3))
plt.subplot(131)
plt.title('All Subjects')
ax = sns.boxplot(x='population', y='age', data=demographics, width=0.9, fliersize=0)
ax = sns.swarmplot(x='population', y='age', data=demographics, color='red', edgecolor='black', linewidth=1, size=2)
ax.set_ylim([20,105])
plt.subplot(132)
plt.title('Female Subjects')
ax = sns.boxplot(x='population', y='age', data=demographics[demographics.sex==0], width=0.9, fliersize=0)
ax = sns.swarmplot(x='population', y='age', data=demographics[demographics.sex==0], color='red', edgecolor='black', linewidth=1, size=2)
ax.set_ylim([20,105])

plt.subplot(133)
plt.title('Male Subjects')
ax = sns.boxplot(x='population', y='age', data=demographics[demographics.sex==1], width=0.9, fliersize=0)
ax = sns.swarmplot(x='population', y='age', data=demographics[demographics.sex==1], color='red', edgecolor='black', linewidth=1, size=2)
ax.set_ylim([20,105])
plt.tight_layout()


sel = demographics.loc[(demographics.sex==0) & (demographics.population=='ICU'), 'age']
n_female_icu = sel.shape[0]
sel = demographics.loc[(demographics.sex==0) & (demographics.population=='Sleeplab'), 'age']
n_female_sleeplab = sel.shape[0]

sel = demographics.loc[(demographics.sex==1) & (demographics.population=='ICU'), 'age']
n_male_icu = sel.shape[0]
sel = demographics.loc[(demographics.sex==1) & (demographics.population=='Sleeplab'), 'age']
n_male_sleeplab = sel.shape[0]
print('')
print(f'n_female_icu:      {n_female_icu}')
print(f'n_female_sleeplab: {n_female_sleeplab}')
print(f'n_male_icu:        {n_male_icu}')
print(f'n_male_sleeplab:   {n_male_sleeplab}')
print('')
test_samples = []
sel = demographics.loc[demographics.population=='ICU', 'age']
test_samples.append(sel.values)
print(f"Median age for ICU:             {sel.median():.1f} , IQR: [{sel.quantile(0.25):.1f}, {sel.quantile(0.75):.1f}]")
sel = demographics.loc[demographics.population=='Sleeplab', 'age']
test_samples.append(sel.values)
print(f"Median age for sleeplab:        {sel.median():.1f} , IQR: [{sel.quantile(0.25):.1f}, {sel.quantile(0.75):.1f}]")
t_statistic, p_value = ttest_ind(test_samples[0], test_samples[1], equal_var=False)
print(f't-test (Welch): p-value:        {p_value:.2f}')
t_statistic, p_value = ttest_ind(test_samples[0], test_samples[1])
print(f'Mann Whiteney U test: p-value:  {p_value:.2f}')

print('')
test_samples = []
sel = demographics.loc[(demographics.sex == 0 ) & (demographics.population=='ICU'), 'age']
test_samples.append(sel.values)
print(f"Median age for ICU female:      {sel.median():.1f} , IQR: [{sel.quantile(0.25):.1f}, {sel.quantile(0.75):.1f}]")
sel = demographics.loc[(demographics.sex == 0) & (demographics.population=='Sleeplab'), 'age']
test_samples.append(sel.values)
print(f"Median age for sleeplab female: {sel.median():.1f} , IQR: [{sel.quantile(0.25):.1f}, {sel.quantile(0.75):.1f}]")
t_statistic, p_value = ttest_ind(test_samples[0], test_samples[1], equal_var=False)
print(f't-test (Welch): p-value:        {p_value:.2f}')
t_statistic, p_value = ttest_ind(test_samples[0], test_samples[1])
print(f'Mann Whiteney U test: p-value:  {p_value:.2f}')
test_samples = []
sel = demographics.loc[(demographics.sex == 1) & (demographics.population=='ICU'), 'age']
test_samples.append(sel.values)
print(f"\nMedian age for ICU male:        {sel.median():.1f} , IQR: [{sel.quantile(0.25):.1f}, {sel.quantile(0.75):.1f}]")
sel = demographics.loc[(demographics.sex == 1) & (demographics.population=='Sleeplab'), 'age']
test_samples.append(sel.values)
print(f"Median age for sleeplab male:   {sel.median():.1f} , IQR: [{sel.quantile(0.25):.1f}, {sel.quantile(0.75):.1f}]")
t_statistic, p_value = ttest_ind(test_samples[0], test_samples[1], equal_var=False)
print(f't-test (Welch): p-value:        {p_value:.2f}')
t_statistic, p_value = ttest_ind(test_samples[0], test_samples[1])
print(f'Mann Whiteney U test: p-value:  {p_value:.2f}')

print(f"\nMale/Female Ratio ICU:          {n_male_icu / n_female_icu:.2f}")
print(f"Male/Female Ratio Sleeplab:     {n_male_sleeplab / n_female_sleeplab:.2f}")

print('')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …


n_female_icu:      41
n_female_sleeplab: 18
n_male_icu:        62
n_male_sleeplab:   34

Median age for ICU:             68.0 , IQR: [62.5, 75.0]
Median age for sleeplab:        70.5 , IQR: [60.5, 74.8]
t-test (Welch): p-value:        0.51
Mann Whiteney U test: p-value:  0.49

Median age for ICU female:      67.0 , IQR: [57.0, 76.0]
Median age for sleeplab female: 71.8 , IQR: [67.3, 76.1]
t-test (Welch): p-value:        0.19
Mann Whiteney U test: p-value:  0.15

Median age for ICU male:        68.0 , IQR: [64.0, 75.0]
Median age for sleeplab male:   69.7 , IQR: [60.1, 74.0]
t-test (Welch): p-value:        0.56
Mann Whiteney U test: p-value:  0.56

Male/Female Ratio ICU:          1.51
Male/Female Ratio Sleeplab:     1.89



In [104]:
study_ids_matched = demographics.loc[demographics.population == 'Sleeplab', 'study_id'].values

In [105]:
summary_days_sleeplab_all.loc[np.isin(summary_days_sleeplab_all.study_id.values, study_ids_matched),
                                      'matched_' + sleeplab_cohort] = 1

In [106]:
### end of manual loop.

In [107]:
summary_days_sleeplab_all.columns[-6:]

Index(['arousali_expert_agreement_relaxed',
       'hours_data_expert_disagreement_relaxed',
       'hours_sleep_expert_disagreement_relaxed',
       'perc_W_expert_disagreement_relaxed',
       'perc_S_expert_disagreement_relaxed',
       'perc_R_expert_disagreement_relaxed'],
      dtype='object')

In [108]:
# change columns order:
new_col_order = np.concatenate([summary_days_sleeplab_all.columns[:5].values, summary_days_sleeplab_all.columns[-6:].values, summary_days_sleeplab_all.columns[5:-6]])
summary_days_sleeplab_all = summary_days_sleeplab_all[new_col_order]

In [109]:
summary_days_sleeplab_all

,study_id,mrn,age,sex,cci,arousali_expert_agreement_relaxed,hours_data_expert_disagreement_relaxed,hours_sleep_expert_disagreement_relaxed,perc_W_expert_disagreement_relaxed,perc_S_expert_disagreement_relaxed,...,hours_data_expert_agreement_relaxed,hours_sleep_expert_agreement_relaxed,perc_W_expert_agreement_relaxed,perc_S_expert_agreement_relaxed,perc_R_expert_agreement_relaxed,perc_N1_expert_agreement_relaxed,perc_N2_expert_agreement_relaxed,perc_N3_expert_agreement_relaxed,sfi_expert_agreement_relaxed,sfi_w_expert_agreement_relaxed
3,4,2676815.0,46.975,Male,0.0,3.9,0.808334,0.808334,0.000000,0.999999,...,6.816668,6.216668,0.088020,0.911980,0.194370,0.058981,0.475871,0.270777,4.5,3.1
4,5,3966502.0,49.192,Male,0.0,6.5,5.358334,5.358334,0.000000,1.000000,...,0.991668,0.766668,0.226891,0.773108,0.054348,0.315217,0.619564,0.010870,10.4,2.6
5,6,2218765.0,73.466,Male,3.0,7.0,1.516668,1.041668,0.313187,0.686813,...,5.233334,4.025001,0.230892,0.769108,0.155279,0.159420,0.656315,0.028986,5.2,3.2
6,7,4560083.0,61.797,Male,0.0,3.3,2.608334,2.525001,0.031949,0.968051,...,4.450001,3.341668,0.249064,0.750936,0.204489,0.087282,0.690773,0.017456,2.7,1.5
7,11,4993308.0,72.638,Female,4.0,0.4,1.491668,1.483334,0.005587,0.994413,...,5.500001,5.166668,0.060606,0.939394,0.190323,0.003226,0.296774,0.509677,0.4,0.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405,458,3544572.0,69.022,Male,1.0,4.9,0.016668,0.016668,0.000000,0.999940,...,6.916668,2.441668,0.646988,0.353012,0.037543,0.354949,0.576792,0.000000,13.5,3.7
406,460,4567135.0,56.068,Male,0.0,3.5,0.466668,0.466668,0.000000,0.999998,...,7.583334,6.658334,0.121978,0.878022,0.117647,0.167710,0.609512,0.088861,4.2,1.5
407,461,3619992.0,64.510,Male,1.0,2.0,0.075001,0.075001,0.000000,0.999987,...,5.716668,5.066668,0.113703,0.886297,0.000000,0.083882,0.521381,0.388158,1.2,0.6
410,464,4777940.0,69.216,Male,1.0,2.5,0.283334,0.275001,0.029412,0.970585,...,5.500001,4.050001,0.263636,0.736364,0.121399,0.037037,0.460905,0.290123,2.2,2.2


In [26]:
summary_days_sleeplab_all

,study_id,mrn,age,sex,cci,perc_N1_expert_disagreement_relaxed,perc_N2_expert_disagreement_relaxed,perc_N3_expert_disagreement_relaxed,sfi_expert_disagreement_relaxed,sfi_w_expert_disagreement_relaxed,...,perc_N2_expert_agreement_relaxed,perc_N3_expert_agreement_relaxed,sfi_expert_agreement_relaxed,sfi_w_expert_agreement_relaxed,arousali_expert_agreement_relaxed,hours_data_expert_disagreement_relaxed,hours_sleep_expert_disagreement_relaxed,perc_W_expert_disagreement_relaxed,perc_S_expert_disagreement_relaxed,perc_R_expert_disagreement_relaxed
3,4,2676815.0,46.975,Male,0.0,0.010309,0.773195,0.020619,1.2,0.0,...,0.475871,0.270777,4.5,3.1,3.9,0.808334,0.808334,0.000000,0.999999,0.195876
4,5,3966502.0,49.192,Male,0.0,0.003110,0.628305,0.166407,0.4,0.0,...,0.619564,0.010870,10.4,2.6,6.5,5.358334,5.358334,0.000000,1.000000,0.202177
5,6,2218765.0,73.466,Male,3.0,0.152000,0.416000,0.000000,6.7,4.8,...,0.656315,0.028986,5.2,3.2,7.0,1.516668,1.041668,0.313187,0.686813,0.432000
6,7,4560083.0,61.797,Male,0.0,0.036304,0.623762,0.161716,2.8,2.4,...,0.690773,0.017456,2.7,1.5,3.3,2.608334,2.525001,0.031949,0.968051,0.178218
7,11,4993308.0,72.638,Female,4.0,0.011236,0.601123,0.073034,2.0,0.7,...,0.296774,0.509677,0.4,0.4,0.4,1.491668,1.483334,0.005587,0.994413,0.314607
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405,458,3544572.0,69.022,Male,1.0,0.499970,0.499970,0.000000,60.0,0.0,...,0.576792,0.000000,13.5,3.7,4.9,0.016668,0.016668,0.000000,0.999940,0.000000
406,460,4567135.0,56.068,Male,0.0,0.017857,0.607142,0.017857,2.1,0.0,...,0.609512,0.088861,4.2,1.5,3.5,0.466668,0.466668,0.000000,0.999998,0.357142
407,461,3619992.0,64.510,Male,1.0,0.000000,0.999987,0.000000,0.0,0.0,...,0.521381,0.388158,1.2,0.6,2.0,0.075001,0.075001,0.000000,0.999987,0.000000
410,464,4777940.0,69.216,Male,1.0,0.030303,0.393938,0.000000,3.6,3.6,...,0.460905,0.290123,2.2,2.2,2.5,0.283334,0.275001,0.029412,0.970585,0.575755


In [98]:
dir_summaries = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing'

summary_days_sleeplab_all.to_csv(os.path.join(dir_summaries, 'summary_days_sleeplab_matched.csv'), index=False)

In [92]:
if 0:
    icu_sleep_airgo_demographics = demographics_original[demographics_original.population=='ICU']
    sleeplab_demographics = demographics_original[demographics_original.population=='Sleeplab']

    fig, ax = plt.subplots(figsize=(6,6))
    plt.scatter(sleeplab_demographics['sex'], sleeplab_demographics['age'], alpha=0.3, label='Sleeplab All', s=8)
    plt.scatter(icu_sleep_airgo_demographics['sex']+0.2, icu_sleep_airgo_demographics['age'], alpha=0.4, label='ICU', s=8)
    plt.scatter(matched_df['sex']+0.1, matched_df['age'], marker='x', alpha=0.4, label='Sleeplab Matched', s=8)
    plt.legend()
    plt.xlim(-0.2,1.4)
    # plt.ylim(50,60)

In [ ]:
# code below not used in most recent version.

### save sleeplab airgo demographics selection:

In [61]:
demographics_final_airgo = demographics.copy()
demographics_final_airgo.loc[~pd.isna(demographics_final_airgo.study_id), 'study_id'] = demographics_final_airgo.loc[~pd.isna(demographics_final_airgo.study_id), 'study_id'].apply(lambda x: 'ICU'+str(int(x)))
demographics_final_airgo.loc[pd.isna(demographics_final_airgo.study_id), 'study_id'] = demographics_final_airgo.loc[pd.isna(demographics_final_airgo.study_id), 'SID']
demographics_final_airgo.drop(['SID'], axis=1, inplace=True)


In [63]:
demographics_final_airgo

,Age,MRN,Population,Sex,study_id,Sex_int
0,81.9,183939.0,Sleeplab,Male,Air303,True
1,78.6,405144.0,Sleeplab,Male,Air087,True
3,75.3,658243.0,Sleeplab,Male,Air448,True
4,77.0,668419.0,Sleeplab,Male,Air053,True
5,68.2,780070.0,Sleeplab,Male,Air148,True
...,...,...,...,...,...,...
529,67.0,NaN,ICU,Female,ICU185,False
530,79.0,NaN,ICU,Female,ICU186,False
531,71.0,NaN,ICU,Male,ICU187,True
532,62.0,NaN,ICU,Female,ICU188,False


In [64]:
demographics_final_airgo.to_csv('Matched_Cohort_ICU_Sleeplab_withAirgo.csv', index=False)

### sleeplab full

In [65]:
sleeplab_full['Sex_int'] = sleeplab_full['Sex'] == 'Male'
sleeplab_full['Sex_int'] = sleeplab_full['Sex_int'].astype(int)
sleeplab_full['Age'] = sleeplab_full['Age_at_Visit']
sleeplab_full = sleeplab_full.dropna(subset=['Age', 'Sex_int'])

In [66]:
icu_sleep_airgo_demographics['Population'] = 'ICU'
sleeplab_full['Population'] = 'Sleeplab'
icu_sleep_airgo_demographics['Sex_int'] = (icu_sleep_airgo_demographics['Sex'] == 'Male').astype(int)

demographics_original = pd.concat([sleeplab_full, icu_sleep_airgo_demographics], axis=0)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:5: FutureWarnin

In [67]:
demographics_original.Age

0      39.123
1      66.384
2      65.619
3      71.247
4      50.674
        ...  
185    67.000
186    79.000
187    71.000
188    62.000
189    76.000
Name: Age, Length: 24584, dtype: float64

In [68]:
variables = ['Age','Sex_int']
sleeplab_full[variables]

,Age,Sex_int
0,39.123,0
1,66.384,0
2,65.619,0
3,71.247,0
4,50.674,0
...,...,...
24461,57.068,1
24462,57.605,0
24463,56.186,0
24464,72.227,0


In [69]:
variables = ['Age','Sex_int']

scale_age = 0.1
n_neighbors = 75


print(f'n_neighbor: {n_neighbors}')
demographics = demographics_original.copy()   

#     print(demographics['Age'].iloc[:3])
#     print(demographics['Age'].iloc[-3:])

#     scale_age = 1000
demographics['Age'] = demographics['Age'].astype(float).values/scale_age
#     print(demographics['Age'].iloc[:3])
#     print(demographics['Age'].iloc[-3:])

demographics.reset_index(drop=True, inplace=True)
#     demographics['Sex_int'] = demographics['Sex_int'].astype(int)

icu_sleep_airgo_demographics = demographics[demographics.Population=='ICU'].copy()
sleeplab_demographics = demographics[demographics.Population=='Sleeplab'].copy()

#     print(icu_sleep_airgo_demographics[variables].head(3))
#     print(sleeplab_demographics[variables].head(3))


to_match_df = icu_sleep_airgo_demographics[variables]
to_match_df = to_match_df[to_match_df.Age>=(50/scale_age)]

#     print(to_match_df.head(3))
#     print(sleeplab_demographics[variables].head(3))
matched_df = get_matching_pairs(to_match_df, sleeplab_demographics[variables], n_neighbors=n_neighbors, scaler=False)
print(f'No. of matches: {matched_df.shape[0]}')

#     print(matched_df.head(3))

idx_keep = np.concatenate([np.array(matched_df.index), np.array(icu_sleep_airgo_demographics.index)])
no_matched = len(matched_df.index)
#     print(f'No matched: {no_matched}')
demographics = demographics.loc[idx_keep]
demographics.shape
demographics['Age'] = demographics['Age']*scale_age

sel = demographics.loc[(demographics.Sex=='Female') & (demographics.Population=='ICU'), 'Age']
n_female_icu = sel.shape[0]
sel = demographics.loc[(demographics.Sex=='Female') & (demographics.Population=='Sleeplab'), 'Age']
n_female_sleeplab = sel.shape[0]

sel = demographics.loc[(demographics.Sex=='Male') & (demographics.Population=='ICU'), 'Age']
n_male_icu = sel.shape[0]
sel = demographics.loc[(demographics.Sex=='Male') & (demographics.Population=='Sleeplab'), 'Age']
n_male_sleeplab = sel.shape[0]

print(n_female_icu)
print(n_female_sleeplab)
print(n_male_icu)
print(n_male_sleeplab)

sel = demographics.loc[(demographics.Sex=='Female') & (demographics.Population=='ICU'), 'Age']
print(f"Median Age for ICU female:      {sel.median():.1f} , IQR: [{sel.quantile(0.25)}, {sel.quantile(0.75)}]")
sel = demographics.loc[(demographics.Sex=='Female') & (demographics.Population=='Sleeplab'), 'Age']
print(f"Median Age for sleeplab female: {sel.median():.1f} , IQR: [{sel.quantile(0.25)}, {sel.quantile(0.75)}]")

sel = demographics.loc[(demographics.Sex=='Male') & (demographics.Population=='ICU'), 'Age']
print(f"Median Age for ICU male:        {sel.median():.1f} , IQR: [{sel.quantile(0.25)}, {sel.quantile(0.75)}]")
sel = demographics.loc[(demographics.Sex=='Male') & (demographics.Population=='Sleeplab'), 'Age']
print(f"Median Age for sleeplab male:   {sel.median():.1f} , IQR: [{sel.quantile(0.25)}, {sel.quantile(0.75)}]")
print(f"Male/Female Ratio ICU:      {n_male_icu / n_female_icu:.2f}")
print(f"Male/Female Ratio Sleeplab: {n_male_sleeplab / n_female_sleeplab:.2f}")

print('')




n_neighbor: 75
No. of matches: 3157
53
1331
77
1796
Median Age for ICU female:      66.0 , IQR: [56.0, 74.0]
Median Age for sleeplab female: 66.0 , IQR: [56.907, 73.982]
Median Age for ICU male:        67.0 , IQR: [62.0, 73.0]
Median Age for sleeplab male:   67.1 , IQR: [60.00375, 75.91275]
Male/Female Ratio ICU:      1.45
Male/Female Ratio Sleeplab: 1.35



In [72]:
demographics.to_csv('Matched_Cohort_ICU_Sleeplab_withoutAirgo.csv', index=False)

In [276]:
for i in range(1000):
    plt.close()